# 06 — Interpretation & evaluation
Lead-time skill, physics-feature ablation, permutation importance, cost–loss value, LOSO transfer. Writes `interpret_metrics.json`.

In [ ]:
# === Colab/local auto-setup (device + data path) ===
import sys, os, subprocess
from pathlib import Path
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split('==')[0].replace('-', '_').replace('scikit_learn', 'sklearn')
        try:
            __import__(mod)
        except Exception:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
_pip('numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'torch', 'matplotlib')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
CSV = 'Bangladesh Meterological data.csv'
cands = [Path.cwd()/CSV, Path('/content')/CSV, Path(r'd:\BUET RESEARCH WORK\Bangladesh Flood')/CSV]
root = next((c.parent for c in cands if c.exists()), None)
if root is None:
    try:
        from google.colab import files
        files.upload(); root = Path.cwd()
    except Exception:
        raise FileNotFoundError('Upload "%s" next to this notebook.' % CSV)
os.environ['DFAA_ROOT'] = str(root)
print('DFAA_ROOT =', root)


In [ ]:
%%writefile common.py
"""Common config, data loader, and leak-free helpers for the Bangladesh DFAA study.

All paths hardcoded (workspace convention). Train-only fits everywhere.
Notation matches EXPERIMENT_DESIGN.md. No experiment numbers are produced here;
this is the shared, smoke-testable core that the notebooks reuse.
"""
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# ROOT overridable so the shipped notebooks run on Colab/local (set env DFAA_ROOT).
ROOT = Path(os.environ.get("DFAA_ROOT", r"d:\BUET RESEARCH WORK\Bangladesh Flood"))
RAW_CSV = ROOT / "Bangladesh Meterological data.csv"
ART = ROOT / "artefacts"
ART.mkdir(exist_ok=True)

# ---- locked study constants (EXPERIMENT_DESIGN.md defaults) ----
SEED = 0
VARS_Z = ["Rainfall_mm", "Soil_moisture_mm"]          # standardized by train climatology
# robust standardization: per-(s,m) sd floored at SD_FLOOR_FRAC * station-pooled train sd,
# then z clipped to +-Z_CLIP. Guards the soil-moisture saturation / dry-month near-zero-sd
# pathology (otherwise z -> ~-40000). Fixed/train-only transforms => no leakage; train cells
# (|z|<3.6) are untouched. Documented in RESULTS_LOG S1/S2.
SD_FLOOR_FRAC = 0.15
Z_CLIP = 4.0
W_WEIGHTS = (1 / 3, 1 / 3, 1 / 3)                     # w1*z_P + w2*z_SM + w3*SPEI
ALPHA_DFAA = 1.8                                       # Wu (2006) constant in Eq. 2
LEADS = (1, 2, 3)                                      # symmetric window scale = lead h
TAUS = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95], dtype=np.float64)
THETA_PCT = 80.0                                       # theta_D = 80th pct of |DFAA| on train

# time-ordered split by ORIGIN year (the month t at which DFAA(s,t) is anchored)
TRAIN_YEARS = (2000, 2014)
VAL_YEARS = (2015, 2017)
TEST_YEARS = (2018, 2022)

# BMD station latitudes/longitudes (deg) for PET extraterrestrial radiation + maps.
# Standard BMD station coordinates; for PET only latitude matters (Ra is ~flat to +-0.2 deg).
# Provenance flagged for final verification before the .tex (CLAUDE.md Rule 3).
STATION_LATLON = {
    "Barisal":              (22.70, 90.37),
    "Bogra":                (24.85, 89.37),
    "Chittagong(Air-port)": (22.25, 91.81),
    "Comilla":              (23.43, 91.18),
    "Cox's Bazar":          (21.45, 91.97),
    "Dhaka":                (23.78, 90.38),
    "Faridpur":             (23.60, 89.85),
    "Jessore":              (23.18, 89.16),
    "Khulna":               (22.78, 89.53),
    "Mymensingh":           (24.75, 90.43),
    "Rajshahi":             (24.37, 88.70),
    "Rangpur":              (25.73, 89.23),
    "Sylhet":               (24.90, 91.88),
}


def load_clean():
    """Load raw CSV, drop the trailing all-NaN row, sort, add integer month index t.

    Returns a tidy long DataFrame with columns:
      Station_Name, Station_Code, Year, Month, Max_Temp, Min_Temp, Rainfall_mm,
      Soil_moisture_mm, SPEI_3, s (0..12 station id), t (0-based global month index),
      origin_year, split.
    Raw file is never modified.
    """
    df = pd.read_csv(RAW_CSV)
    # drop rows that are entirely NaN in the value columns (the trailing NaN row)
    val_cols = ["Max_Temp", "Min_Temp", "Rainfall_mm", "Soil_moisture_mm", "SPEI_3"]
    before = len(df)
    df = df.dropna(subset=["Station_Name", "Year", "Month"], how="any").copy()
    df = df.dropna(subset=val_cols, how="all").copy()
    dropped = before - len(df)

    df["Year"] = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)
    df = df.sort_values(["Station_Name", "Year", "Month"]).reset_index(drop=True)

    stations = sorted(df["Station_Name"].unique())
    sid = {name: i for i, name in enumerate(stations)}
    df["s"] = df["Station_Name"].map(sid)

    # global 0-based month index over 2000-01 .. 2022-12
    df["t"] = (df["Year"] - 2000) * 12 + (df["Month"] - 1)

    def split_of(y):
        if TRAIN_YEARS[0] <= y <= TRAIN_YEARS[1]:
            return "train"
        if VAL_YEARS[0] <= y <= VAL_YEARS[1]:
            return "val"
        return "test"

    df["origin_year"] = df["Year"]
    df["split"] = df["Year"].map(split_of)
    return df, stations, sid, dropped


def to_grid(df, col):
    """Return an (S, T) float array of `col` indexed by [station s, month index t],
    NaN where missing. S=13 stations, T=276 months (2000-01..2022-12)."""
    S = df["s"].nunique()
    T = 276
    g = np.full((S, T), np.nan, dtype=np.float64)
    g[df["s"].to_numpy(), df["t"].to_numpy()] = df[col].to_numpy(dtype=float)
    return g


def train_mask_t(years=TRAIN_YEARS):
    """Boolean length-276 mask of month indices whose calendar year is in `years`."""
    t = np.arange(276)
    yr = 2000 + t // 12
    return (yr >= years[0]) & (yr <= years[1])


def fit_climatology(grid, train_t):
    """Per (station s, calendar month m) mean/std on TRAIN months only, with a robust
    std floor at SD_FLOOR_FRAC * station-pooled train sd (guards saturated/dry near-zero-sd
    cells). grid: (S,T); train_t: bool length T. Returns mu,sd as (S,12)."""
    S, T = grid.shape
    mu = np.full((S, 12), np.nan)
    sd = np.full((S, 12), np.nan)
    months = np.arange(T) % 12
    for s in range(S):
        for m in range(12):
            sel = (months == m) & train_t
            vals = grid[s, sel]
            vals = vals[~np.isnan(vals)]
            if len(vals) >= 2:
                mu[s, m] = vals.mean()
                sd[s, m] = vals.std(ddof=1)
    glob = np.nanstd(grid[:, train_t])
    for s in range(S):
        stat_sd = np.nanstd(grid[s, train_t])
        floor = SD_FLOOR_FRAC * stat_sd if np.isfinite(stat_sd) and stat_sd > 1e-6 else glob
        floor = max(floor, 1e-6)
        for m in range(12):
            if not np.isfinite(sd[s, m]) or sd[s, m] < floor:
                sd[s, m] = floor
            if not np.isfinite(mu[s, m]):
                mu[s, m] = np.nanmean(grid[s, train_t])
    return mu, sd


def standardize(grid, mu, sd):
    """z_v(s,t) = clip( (x - mu[s,m]) / sd[s,m], -Z_CLIP, +Z_CLIP ), m = t%12. NaNs propagate."""
    S, T = grid.shape
    months = np.arange(T) % 12
    z = (grid - mu[:, months]) / sd[:, months]
    return np.clip(z, -Z_CLIP, Z_CLIP)


In [ ]:
%%writefile evalkit.py
"""Shared probabilistic-forecast evaluation (reused by Steps 3-6).

All metrics take predictive quantiles Q[N, nT] (monotone non-decreasing in tau) and targets y[N].
CRPS uses the quantile estimator CRPS ~= 2*mean_tau(pinball_tau) (EXPERIMENT_DESIGN Step 7);
coarse but identical across methods, so CRPSS comparisons are fair.
"""
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

from common import TAUS


def pinball_per_tau(Q, y, taus=TAUS):
    err = y[:, None] - Q                       # [N,nT]
    pb = np.maximum(taus[None, :] * err, (taus[None, :] - 1) * err)
    return pb.mean(axis=0)                      # [nT]


def pinball(Q, y, taus=TAUS):
    return float(pinball_per_tau(Q, y, taus).mean())


def crps(Q, y, taus=TAUS):
    return float(2.0 * pinball_per_tau(Q, y, taus).mean())


def coverage(Q, y, lo_i, hi_i):
    lo, hi = Q[:, lo_i], Q[:, hi_i]
    inside = (y >= lo) & (y <= hi)
    return float(inside.mean()), float((hi - lo).mean())


def cdf_at(Q, thr, taus=TAUS):
    """Predictive CDF at threshold thr via linear interpolation across the quantile grid."""
    Q = np.asarray(Q, float)
    N, m = Q.shape
    thr_arr = np.full(N, thr) if np.isscalar(thr) else np.asarray(thr, float)
    k = np.sum(Q < thr_arr[:, None], axis=1)
    lo = np.clip(k - 1, 0, m - 1); hi = np.clip(k, 0, m - 1)
    ar = np.arange(N)
    qlo, qhi = Q[ar, lo], Q[ar, hi]
    tlo, thi = taus[lo], taus[hi]
    gap = qhi - qlo
    frac = np.where(gap > 1e-9, (thr_arr - qlo) / np.where(gap > 1e-9, gap, 1.0), 0.0)
    cdf = tlo + frac * (thi - tlo)
    cdf = np.where(k == 0, taus[0], cdf)
    cdf = np.where(k == m, taus[-1], cdf)
    return np.clip(cdf, 0.0, 1.0)


def event_metrics(Q, y, theta, taus=TAUS):
    """DTF (y>=+theta) and FTD (y<=-theta) detection from the predictive CDF."""
    out = {}
    p_dtf = 1.0 - cdf_at(Q, theta, taus)
    p_ftd = cdf_at(Q, -theta, taus)
    for name, p, ind in [("DTF", p_dtf, (y >= theta).astype(int)),
                          ("FTD", p_ftd, (y <= -theta).astype(int))]:
        d = {"base_rate": float(ind.mean()), "brier": float(brier_score_loss(ind, np.clip(p, 0, 1)))}
        if ind.sum() > 0 and ind.sum() < len(ind):
            d["auc"] = float(roc_auc_score(ind, p))
            d["pr_auc"] = float(average_precision_score(ind, p))
        else:
            d["auc"] = float("nan"); d["pr_auc"] = float("nan")
        out[name] = d
    return out


def all_metrics(Q, y, theta, crps_ref=None, taus=TAUS):
    """Bundle of probabilistic + calibration + event metrics for one method/lead/split."""
    pin = pinball(Q, y, taus)
    cr = crps(Q, y, taus)
    p80, w80 = coverage(Q, y, 1, 5)   # taus index 1=0.10, 5=0.90 -> 80% PI
    p90, w90 = coverage(Q, y, 0, 6)   # taus index 0=0.05, 6=0.95 -> 90% PI
    m = {"n": int(len(y)), "pinball": pin, "crps": cr,
         "picp80": p80, "width80": w80, "picp90": p90, "width90": w90}
    if crps_ref is not None and crps_ref > 0:
        m["crpss"] = float(1.0 - cr / crps_ref)
    m["event"] = event_metrics(Q, y, theta, taus)
    return m


def residual_quantiles(resid_train, taus=TAUS):
    """Empirical quantiles of training residuals -> additive spread for a point forecast."""
    r = resid_train[np.isfinite(resid_train)]
    return np.quantile(r, taus)


def point_to_quantiles(point, resid_q):
    """Q[N,nT] = point[:,None] + resid_q[None,:] (homoscedastic residual probabilization)."""
    Q = point[:, None] + resid_q[None, :]
    return np.maximum.accumulate(Q, axis=1)   # enforce monotone (sorted resid_q already monotone)


In [ ]:
"""Step 6 - interpretation & evaluation.

A. Lead-time skill collation (CRPSS vs h: GBQ / bucket / LSTM).
B. Physics-guided feature ABLATION (GBQ full vs no-physics) - the de-risking decision.
C. Permutation importance (GBQ, pinball-based) - driver attribution.
D. Cost-loss decision value V(r) (GBQ+CQR event probs) - the adaptation/services lens.
E. LOSO transfer (GBQ, h=2) - per-held-out-station skill ("ungauged" angle).

Writes artefacts/interpret_metrics.json + appends RESULTS_LOG.
"""
import json
import time
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from common import ART, load_clean, train_mask_t, LEADS, TAUS, SEED
import evalkit as ek

np.random.seed(SEED)
print("=" * 70); print("STEP 6 - interpretation & evaluation"); print("=" * 70)

df, stations, sid, _ = load_clean()
S, T = len(stations), 276
train_t = train_mask_t()
yr = 2000 + np.arange(T) // 12

fz = np.load(ART / "features.npz", allow_pickle=True)
feat = fz["feat"].astype(np.float64); fnames = list(fz["names"]); Fdim = feat.shape[-1]
dz = np.load(ART / "dfaa.npz", allow_pickle=True)
DFAA = {h: dz[f"DFAA_h{h}"] for h in LEADS}; WE = {h: dz[f"WE_h{h}"] for h in LEADS}
meta = json.loads((ART / "dfaa_meta.json").read_text()); THETA = {h: meta["theta_D"][str(h)] for h in LEADS}
base_m = json.loads((ART / "baseline_metrics.json").read_text())
model_m = json.loads((ART / "model_metrics.json").read_text())
conf_m = json.loads((ART / "conformal_metrics.json").read_text())

fmu = np.array([np.nanmean(feat[:, train_t, j]) for j in range(Fdim)])
feat_fill = np.where(np.isfinite(feat), feat, fmu[None, None, :])
we_mu = {h: float(np.nanmean(WE[h][:, train_t])) for h in LEADS}
PHYS = ["PET", "D", "aridity", "API", "dSM", "dSPEI", "roll3", "roll6"]
phys_idx = [fnames.index(n) for n in PHYS]
allcols = list(range(Fdim)) + ["WE"]   # WE appended as last column


def split_of(t):
    y = yr[t]; return "train" if y <= 2014 else ("val" if y <= 2017 else "test")


def build_Xy(h, idx, drop_cols=()):
    rows = []
    for s, t in idx:
        v = list(feat_fill[s, t]) + [WE[h][s, t] if np.isfinite(WE[h][s, t]) else we_mu[h]]
        rows.append(v)
    X = np.array(rows)
    if drop_cols:
        keep = [c for c in range(X.shape[1]) if c not in drop_cols]
        X = X[:, keep]
    y = np.array([DFAA[h][s, t] for s, t in idx])
    return X, y


def origins(h, sp):
    return [(s, t) for s in range(S) for t in range(T) if np.isfinite(DFAA[h][s, t]) and split_of(t) == sp]


def fit_gbq(Xtr, ytr, Xte, n_est=300):
    cols = []
    for tau in TAUS:
        gb = GradientBoostingRegressor(loss="quantile", alpha=float(tau), n_estimators=n_est,
                                       max_depth=3, learning_rate=0.05, subsample=0.8, random_state=SEED)
        gb.fit(Xtr, ytr); cols.append(gb.predict(Xte))
    return np.sort(np.stack(cols, 1), 1)


out = {}

# ---------- A. lead-time skill ----------
out["lead_time"] = {"gbq_crpss": {h: model_m["gbq"][str(h)]["test"]["crpss"] for h in LEADS},
                    "lstm_crpss": {h: model_m["lstm"][str(h)]["test"]["crpss"] for h in LEADS},
                    "bucket_crpss": {h: base_m[str(h)]["test"]["bucket"]["crpss"] for h in LEADS},
                    "gbq_dtf_auc": {h: model_m["gbq"][str(h)]["test"]["event"]["DTF"]["auc"] for h in LEADS},
                    "gbq_ftd_auc": {h: model_m["gbq"][str(h)]["test"]["event"]["FTD"]["auc"] for h in LEADS}}
print("\n[A] Lead-time CRPSS (test): GBQ / bucket / LSTM")
for h in LEADS:
    print(f"  h={h}  GBQ {out['lead_time']['gbq_crpss'][h]:+.3f} | bucket "
          f"{out['lead_time']['bucket_crpss'][h]:+.3f} | LSTM {out['lead_time']['lstm_crpss'][h]:+.3f}")

# ---------- B. physics-feature ablation ----------
print("\n[B] Physics-guided feature ablation (GBQ full vs no-physics):")
out["ablation"] = {}
for h in LEADS:
    tr, te = origins(h, "train"), origins(h, "test")
    ytr = np.array([DFAA[h][s, t] for s, t in tr]); yte = np.array([DFAA[h][s, t] for s, t in te])
    ref = ek.crps(np.tile(np.quantile(ytr, TAUS), (len(yte), 1)), yte)
    Xtr_f, _ = build_Xy(h, tr); Xte_f, _ = build_Xy(h, te)
    Xtr_n, _ = build_Xy(h, tr, drop_cols=phys_idx); Xte_n, _ = build_Xy(h, te, drop_cols=phys_idx)
    Qf = fit_gbq(Xtr_f, ytr, Xte_f); Qn = fit_gbq(Xtr_n, ytr, Xte_n)
    mf = ek.all_metrics(Qf, yte, THETA[h], crps_ref=ref); mn = ek.all_metrics(Qn, yte, THETA[h], crps_ref=ref)
    out["ablation"][h] = {"full": mf, "no_phys": mn}
    print(f"  h={h}  CRPSS full {mf['crpss']:+.3f} vs no-phys {mn['crpss']:+.3f}  (d={mf['crpss']-mn['crpss']:+.3f}) | "
          f"DTF-AUC full {mf['event']['DTF']['auc']:.3f} vs {mn['event']['DTF']['auc']:.3f}")

# ---------- C. permutation importance (GBQ, h=2, pinball-based) ----------
print("\n[C] Permutation importance (GBQ h=2, increase in pinball when feature shuffled):")
h = 2
tr, te = origins(h, "train"), origins(h, "test")
ytr = np.array([DFAA[h][s, t] for s, t in tr]); yte = np.array([DFAA[h][s, t] for s, t in te])
Xtr, _ = build_Xy(h, tr); Xte, _ = build_Xy(h, te)
gbs = []
for tau in TAUS:
    gb = GradientBoostingRegressor(loss="quantile", alpha=float(tau), n_estimators=300, max_depth=3,
                                   learning_rate=0.05, subsample=0.8, random_state=SEED).fit(Xtr, ytr)
    gbs.append(gb)
def pinball_of(X):
    Q = np.sort(np.stack([gb.predict(X) for gb in gbs], 1), 1)
    return ek.pinball(Q, yte)
base_pin = pinball_of(Xte)
colnames = fnames + ["WE"]
rng = np.random.default_rng(SEED)
imp = {}
for j in range(Xte.shape[1]):
    deltas = []
    for _ in range(5):
        Xp = Xte.copy(); Xp[:, j] = rng.permutation(Xp[:, j]); deltas.append(pinball_of(Xp) - base_pin)
    imp[colnames[j]] = float(np.mean(deltas))
ranked = sorted(imp.items(), key=lambda kv: -kv[1])
out["perm_importance_h2"] = dict(ranked)
for name, val in ranked[:10]:
    tag = " [physics]" if name in PHYS else ""
    print(f"   {name:10s} Δpinball={val:+.5f}{tag}")

# ---------- D. cost-loss value (GBQ+CQR event probs, h=2) ----------
print("\n[D] Cost-loss decision value V(r) (GBQ+CQR, h=2):")
pp = np.load(ART / "model_preds.npz", allow_pickle=True)
Qcal = pp["gbq_2_val_Q"]; ycal = pp["gbq_2_val_y"]; Qte = pp["gbq_2_test_Q"]; yte2 = pp["gbq_2_test_y"]
# apply CQR (reuse Step 5 logic: widen 90/80/50 pairs)
PAIRS = [(0, 6, 0.10), (1, 5, 0.20), (2, 4, 0.50)]
Qc = Qte.copy()
for lo_i, hi_i, a in PAIRS:
    E = np.maximum(Qcal[:, lo_i] - ycal, ycal - Qcal[:, hi_i])
    k = min(int(np.ceil((len(E) + 1) * (1 - a))), len(E)); e = np.sort(E)[k - 1]
    Qc[:, lo_i] -= e; Qc[:, hi_i] += e
Qc = np.sort(Qc, 1)
th = THETA[2]
p_dtf = 1 - ek.cdf_at(Qc, th); occ_dtf = (yte2 >= th).astype(float)
p_ftd = ek.cdf_at(Qc, -th); occ_ftd = (yte2 <= -th).astype(float)
p_clim_dtf = np.full_like(p_dtf, occ_dtf.mean()); p_clim_ftd = np.full_like(p_ftd, occ_ftd.mean())


def cost_loss_curve(p, occ):
    rs = np.linspace(0.02, 0.98, 49); s = occ.mean(); vals = []
    for r in rs:
        act = (p > r).astype(float)
        E_fc = np.mean(act * r + (1 - act) * occ)
        E_clim = min(r, s); E_perf = s * r; den = E_clim - E_perf
        vals.append((E_clim - E_fc) / den if den > 1e-9 else np.nan)
    return rs, np.array(vals)


out["cost_loss_h2"] = {}
for nm, p, occ in [("DTF", p_dtf, occ_dtf), ("FTD", p_ftd, occ_ftd)]:
    rs, v = cost_loss_curve(p, occ)
    vmax = float(np.nanmax(v)); rstar = float(rs[np.nanargmax(v)])
    out["cost_loss_h2"][nm] = {"r": rs.tolist(), "value": v.tolist(), "vmax": vmax, "r_at_vmax": rstar,
                               "base_rate": float(occ.mean())}
    print(f"   {nm}: base rate {occ.mean():.3f}  max value V={vmax:.3f} at C/L={rstar:.2f}  "
          f"(positive V over r in [{rs[np.where(v>0)[0][0]] if np.any(v>0) else float('nan'):.2f}, "
          f"{rs[np.where(v>0)[0][-1]] if np.any(v>0) else float('nan'):.2f}])")

# ---------- E. LOSO transfer (GBQ, h=2) ----------
print("\n[E] LOSO transfer (GBQ h=2): train on 12 stations, test held-out station (2018-22)...")
t0 = time.time(); out["loso_h2"] = {}
for s_out in range(S):
    tr = [(s, t) for (s, t) in origins(2, "train") if s != s_out]
    te = [(s, t) for (s, t) in origins(2, "test") if s == s_out]
    if len(te) < 10:
        continue
    Xtr, ytr = build_Xy(2, tr); Xte, yte_s = build_Xy(2, te)
    Q = fit_gbq(Xtr, ytr, Xte, n_est=200)
    ref = ek.crps(np.tile(np.quantile(ytr, TAUS), (len(yte_s), 1)), yte_s)
    m = ek.all_metrics(Q, yte_s, THETA[2], crps_ref=ref)
    out["loso_h2"][stations[s_out]] = {"crpss": m["crpss"], "crps": m["crps"],
                                       "dtf_auc": m["event"]["DTF"]["auc"], "ftd_auc": m["event"]["FTD"]["auc"]}
    print(f"   {stations[s_out]:22s} CRPSS {m['crpss']:+.3f}  DTF-AUC {m['event']['DTF']['auc']:.3f}  "
          f"FTD-AUC {m['event']['FTD']['auc']:.3f}")
cr = [v["crpss"] for v in out["loso_h2"].values()]
print(f"   LOSO CRPSS mean {np.mean(cr):+.3f}  (min {np.min(cr):+.3f}, max {np.max(cr):+.3f})  in {time.time()-t0:.0f}s")

(ART / "interpret_metrics.json").write_text(json.dumps(out, indent=2, default=float))
print(f"\nWrote {ART/'interpret_metrics.json'}")
print("\nSTEP 6 OK.")
